# Effective-1D self-consistent Thomas–Fermi/Hartree solver with selectable Coulomb geometry

This notebook is a refactor of `gpt_TF_confinement_gate_sweep_compressibility_upgraded(2).ipynb`.  The numerical Thomas–Fermi/Hartree workflow, charge profiles, Landau-level broadening, chemical-potential constraint, diagnostics, gate-distance sweep, animations, and saved outputs are retained, while reusable functions are moved to `iqh_1d_hartree_neumann_confined_grid_fixed.py`.

Selectable electrostatic kernels:

- `"top_gate"`: the original translationally invariant top-gate image kernel, preserved exactly;
- `"top_open_box"`: grounded top gate plus grounded (Dirichlet) sidewalls at $x=\pm x_w$;
- `"top_gate_neumann"`: grounded top gate plus homogeneous Neumann sidewalls at $x=\pm x_N$.

All lengths are in nm, energies in meV, densities in nm$^{-2}$.

**Neumann convention used here:** surface charges are omitted entirely; the Neumann Green function is used only for the electron-electron Hartree term, while donors retain the ordinary single-top-gate kernel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import replace
from pathlib import Path
import json, dataclasses, shutil
from scipy.ndimage import gaussian_filter1d
from IPython.display import HTML, display

from iqh_1d_hartree_neumann_confined_grid_fixed import (
    Params, VALID_KERNELS,
    make_grid, donor_density_profile, s_density_profile,
    donor_potential_analytic, make_kernel_and_background,
    landau_energies, density_from_potential, filling_from_density,
    average_density, solve_self_consistent, plot_solution,
    make_density_potential_animation, make_filling_animation,
)

plt.rcParams.update({"font.size": 13})

## Parameters and electrostatic geometry

The `Params` defaults are the same as in the original notebook.  The new controls are `KERNEL_KIND` and, for bounded kernels, the side-wall position.

For the Neumann geometry an optional depletion-width parametrization is available:
\[
x_N=x_{N,0}-W_{\rm depl}.
\]
Because the original surface-charge grid extends to about $\pm d_S$, the resolved side wall must remain outside every coordinate used by the bounded Green function.  The module checks this explicitly and raises an informative error instead of silently using an inconsistent geometry.

**Neumann convention used here:** surface charges are omitted entirely; the Neumann Green function is used only for the electron-electron Hartree term, while donors retain the ordinary single-top-gate kernel.

In [ ]:
# ======================================================================
# USER PARAMETERS
# Everything that you normally want to change is collected in this cell.
# Units: lengths [nm], energies [meV], areal densities [nm^-2].
# ======================================================================

# ----------------------------------------------------------------------
# 1. Geometry / grids
# ----------------------------------------------------------------------
D_2DEG_NM = 2000.0              # electronic half-width d
NX = 2001                       # number of electronic x-grid points
GRID_MARGIN = 0.999             # x runs from -margin*d to +margin*d
D_SURFACE_NM = D_2DEG_NM + 30.0           # half-width used for the surface-charge grid

# ----------------------------------------------------------------------
# 2. Material and charge densities
# ----------------------------------------------------------------------
EPS_R = 13.0
N_DONOR_NM2 = 2.8e-3
N_SURFACE_NM2 = .7e-3
N_TARGET_NM2 = 2.1e-3

DONOR_B_FRACTION = 0.999        # donor lateral extent = fraction*d
DONOR_SMOOTH_NM = 0.0           # 0 -> sharp donor edge
DONOR_Z_OFFSET_NM = 90.0        # two donor contributions are evaluated at +/- this offset

# ----------------------------------------------------------------------
# 3. Magnetic field / Landau levels
# ----------------------------------------------------------------------
B_T = .47
G_SPIN = 0.0
N_LL = 14
GAMMA_REL = 0.02
USE_B_SCALING_FOR_GAMMA = True

# ----------------------------------------------------------------------
# 4. Hartree self-consistency
# ----------------------------------------------------------------------
MAX_ITER = 60000
MIX = 1.0e-3
TOL = 1.0e-10
SMOOTH_SIGMA_NM = 0.0

# ----------------------------------------------------------------------
# 5. Coulomb Green function / boundary geometry
# ----------------------------------------------------------------------
KERNEL_KIND = "top_gate"
# Allowed:
#   "top_gate"          : infinite grounded top gate only (original notebook)
#   "top_open_box"      : grounded top gate + Dirichlet sidewalls
#   "top_gate_neumann"  : Neumann sidewalls for ELECTRON Hartree kernel only;
#                         donors use standard top-gate kernel; surface charges OFF

# Dirichlet sidewall x = +/- x_w. Used only for "top_open_box".
TOP_OPEN_BOX_HALFWIDTH_NM = 1e6

# Neumann wall parametrization:
# x_N = NEUMANN_BASE_HALFWIDTH_NM - NEUMANN_DEPLETION_WIDTH_NM
NEUMANN_BASE_HALFWIDTH_NM = D_SURFACE_NM
NEUMANN_DEPLETION_WIDTH_NM = 0.0

# ----------------------------------------------------------------------
# 6. Single-calculation reference gate distance
# ----------------------------------------------------------------------
DIST_REF_NM = 1.0e9

# ----------------------------------------------------------------------
# 7. Gate-distance sweep
# ----------------------------------------------------------------------
SWEEP_DIST_MIN_NM = 1.0e2
SWEEP_DIST_MAX_NM = 1.0e4
SWEEP_N_POINTS = 100
SWEEP_MAX_ITER = 60000
SWEEP_TOL = 8.0e-9
USE_WARM_START = True

# ----------------------------------------------------------------------
# 8. Output / animation
# ----------------------------------------------------------------------
RESULTS_DIR = Path("results")


# ======================================================================
# Construct Params explicitly from the values above.
# No physical/numerical choices are hidden in the module.
# ======================================================================
p = Params(
    d=D_2DEG_NM,
    Nx=NX,
    margin=GRID_MARGIN,
    d_S=D_SURFACE_NM,
    eps_r=EPS_R,
    n_donor=N_DONOR_NM2,
    n_s=N_SURFACE_NM2,
    n_target=N_TARGET_NM2,
    donor_b_fraction=DONOR_B_FRACTION,
    donor_smooth_nm=DONOR_SMOOTH_NM,
    donor_z_offset_nm=DONOR_Z_OFFSET_NM,
    B=B_T,
    g_spin=G_SPIN,
    n_LL=N_LL,
    gamma_rel=GAMMA_REL,
    use_B_scaling=USE_B_SCALING_FOR_GAMMA,
    max_iter=MAX_ITER,
    mix=MIX,
    tol=TOL,
    smooth_sigma_nm=SMOOTH_SIGMA_NM,
)

assert KERNEL_KIND in VALID_KERNELS

if KERNEL_KIND == "top_gate":
    kernel_kwargs = dict(
        kernel_kind=KERNEL_KIND,
        sidewall_halfwidth_nm=None,
        depletion_width_nm=0.0,
    )
elif KERNEL_KIND == "top_open_box":
    kernel_kwargs = dict(
        kernel_kind=KERNEL_KIND,
        sidewall_halfwidth_nm=TOP_OPEN_BOX_HALFWIDTH_NM,
        depletion_width_nm=0.0,
    )
else:
    kernel_kwargs = dict(
        kernel_kind=KERNEL_KIND,
        sidewall_halfwidth_nm=NEUMANN_BASE_HALFWIDTH_NM,
        depletion_width_nm=NEUMANN_DEPLETION_WIDTH_NM,
    )

print(p)
print("kernel:", KERNEL_KIND)
if KERNEL_KIND == "top_open_box":
    print(f"Dirichlet sidewalls: x = +/- {TOP_OPEN_BOX_HALFWIDTH_NM:.3f} nm")
elif KERNEL_KIND == "top_gate_neumann":
    xN = NEUMANN_BASE_HALFWIDTH_NM - NEUMANN_DEPLETION_WIDTH_NM
    print(f"Neumann sidewalls: x_N = {xN:.3f} nm")
    print(f"  base half-width = {NEUMANN_BASE_HALFWIDTH_NM:.3f} nm")
    print(f"  W_depl = {NEUMANN_DEPLETION_WIDTH_NM:.3f} nm")

print(f"lB = {p.lB:.3f} nm")
print(f"hbar omega_c = {p.hbar_omega_c:.3f} meV")
print(f"Gamma = {p.Gamma:.4f} meV")
print(f"Expected average filling = {2*np.pi*p.lB**2*p.n_target:.4f}")
print(f"Same formula = {4.1357*(p.n_target/1e-3)/p.B:.4f}")

## Reference electrostatics and single self-consistent solution

In [ ]:
dist_ref = DIST_REF_NM

x, dx, x_S, K, nD, nS, U_ext, x_D, dx_D = make_kernel_and_background(
    p, dist=dist_ref, verbose=True, **kernel_kwargs
)
UD_anal = donor_potential_analytic(x, p)

fig, ax = plt.subplots(figsize=(7,3))
ax.plot(x*1e-3, U_ext-np.amin(U_ext), label=r"$U_{\rm donor}^{\rm selected}$")
ax.plot(
    x*1e-3,
    UD_anal-np.amin(UD_anal),
    label=r"$U_{\rm donor}^{\rm anal}$",
    linestyle="dashed",
    color="orange",
)
ax.set_ylim(0,1300)
ax.set_xlabel(r"$x$ [$\mu$m]")
ax.set_ylabel("meV")
ax.grid(True)
ax.legend()
fig.tight_layout()

# Charge-profile diagnostic.  In the Neumann case donors live on x_D and
# there is deliberately no surface-charge grid/profile.
fig1, ax1 = plt.subplots(figsize=(7,3))
ax1.plot(x_D, nD/1e-3, label="donors")
if x_S is not None and nS is not None:
    ax1.plot(x_S, nS/1e-3, label="surface charges")
else:
    ax1.axvline(x[0], color="k", ls="--", alpha=0.5, label="2DEG edges")
    ax1.axvline(x[-1], color="k", ls="--", alpha=0.5)
ax1.set_ylim(0,10)
ax1.set_xlabel(r"$x$ [nm]")
ax1.set_ylabel(r"density [$10^{11}$ cm$^{-2}$]")
ax1.grid(True)
ax1.legend()
fig1.tight_layout()

# LL normalization sanity check retained from the original notebook.
for m in [1,2,4,8]:
    E = landau_energies(p)
    mu_test = E[m-1] + 10*p.Gamma
    n_test = density_from_potential(np.zeros_like(x), mu_test, p)
    print("filled spin branches / mean nu:", m, np.mean(filling_from_density(n_test,p)))

res = solve_self_consistent(p, dist=dist_ref, **kernel_kwargs)
print("dist_ref:", dist_ref, "nm")
print("iterations logged:", res["history"][-5:])
print("average electron density:", average_density(res["n"], res["dx"], p), "target:", p.n_target)

# Donor mean is evaluated on the donor grid, not on the confined electron grid.
donor_mean = np.trapz(res["n_donor_profile"], res["x_donor"]) / (
    res["x_donor"][-1] - res["x_donor"][0]
)
print("average donor density on donor grid:", donor_mean)
print("center filling:", res["nu"][len(res["nu"])//2])

plot_solution(res,p)
mu_inf = res["mu"] - np.amin(res["U"])

In [ ]:
U_final = res["U"] - np.amin(res["U"])
print(U_final[0])
plt.plot(res["x"],U_final)

## Sweep over top-gate distance

This retains the original logarithmic sweep and warm-start strategy.  Changing `KERNEL_KIND` changes only the Coulomb Green function used consistently for the Hartree, donor, and surface-charge contributions.

In [ ]:
dist_values = np.geomspace(SWEEP_DIST_MIN_NM, SWEEP_DIST_MAX_NM, SWEEP_N_POINTS)      
p_sweep = replace(p, max_iter=SWEEP_MAX_ITER, tol=SWEEP_TOL)


sweep = []
U_warm = None

for dist in dist_values:
    print(f"solving dist = {dist:8.2f} nm | kernel = {KERNEL_KIND}")

    r = solve_self_consistent(
        p_sweep,
        dist=float(dist),  
        U_initial=(U_warm if USE_WARM_START else None),
        **kernel_kwargs,
    )

    sweep.append(r)

    if USE_WARM_START:
        U_warm = r["U"]

    print(
        f"  center nu = {r['nu'][len(r['nu'])//2]:.4f}, "
        f"mu = {r['mu']:.4f} meV, "
        f"final dn = {r['history'][-1,-1]:.3e}"
    )

x_sweep = sweep[0]["x"]
U_arr = np.array([r["U"] for r in sweep])
n_arr = np.array([r["n"] for r in sweep])
nu_arr = np.array([r["nu"] for r in sweep])
comp_arr = np.array([r["compressibility"] for r in sweep])
mu_arr = np.array([r["mu"] for r in sweep])

print("sweep arrays:", U_arr.shape, n_arr.shape, nu_arr.shape, comp_arr.shape)

In [ ]:
print(U_arr[0,0]-np.amin(U_arr[0,:]))

## Compressibility colormap and animations

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

fig,ax=plt.subplots(figsize=(6,6))
pc=ax.pcolormesh(x_sweep,dist_values,comp_arr/1e-3,shading="auto",vmin=0,vmax=1.75)
ax.set_yscale("log"); ax.set_xlabel(r"$x$ [nm]"); ax.set_xlim(-p.d,0); ax.set_ylabel(r"$d_g$ [nm]")
cbar=fig.colorbar(pc,ax=ax,orientation="horizontal",location="top",pad=.08,fraction=.08)
cbar.set_label(r"$\partial n_e/\partial\mu$ [$10^{11}$ cm$^{-2}$/meV]")
fig.tight_layout(); fig.savefig(RESULTS_DIR/"compressibility_sweep.png",dpi=300)

anim_density,density_video_path=make_density_potential_animation(sweep,p_sweep,outdir=RESULTS_DIR)
anim_filling,filling_video_path=make_filling_animation(sweep,p_sweep,outdir=RESULTS_DIR)
print("density/potential video:",density_video_path)
print("local filling video:",filling_video_path)

display(HTML(anim_density.to_jshtml()))
display(HTML(anim_filling.to_jshtml()))

## Chemical-potential shift and final filling-factor comparison

In [ ]:
font={'family':'serif','color':'black','weight':'normal','size':16}
plt.rcParams['font.family']=font['family']; plt.rcParams['font.size']=font['size']
plt.rcParams['axes.labelsize']=font['size']; plt.rcParams['xtick.labelsize']=font['size']; plt.rcParams['ytick.labelsize']=font['size']

fig,ax=plt.subplots(figsize=(4,3))
mu_all=np.array([(r["mu"]-np.amin(r["U"])) for r in sweep])
curve_mu=gaussian_filter1d(mu_all-mu_inf,sigma=1)
#curve_mu=gaussian_filter1d(mu_all-mu_inf,sigma=.5)
ax.plot(dist_values,curve_mu,color='green',lw=4)
ax.set_xlabel(r'$d_g$ - [nm]'); ax.set_ylabel(r'$\Delta \mu$ - [meV]')
ax.set_xlim(1e2,1e4); ax.set_ylim(-2.,1); ax.set_yticks(np.arange(-2,1,.5)); ax.set_xscale('log'); ax.grid()
fig.tight_layout(); fig.savefig(RESULTS_DIR/"chemical_pot_sweep.pdf")

nu=nu_arr[-1]
n_d=np.argmin(np.abs(dist_values-150)); nuf=nu_arr[n_d]
plt.figure(figsize=(4,4)); plt.plot(x_sweep,nu,lw=3,color='blue'); plt.plot(x_sweep,nuf,lw=3,color='orange')
plt.xlabel(r"$x$ [nm]"); plt.ylabel(r"$\nu(x)=2\pi l_B^2 n(x)$"); plt.grid(True)
plt.xlim(p.d*.0,p.d); plt.ylim(0,14); plt.yticks(np.arange(0,16,2)); plt.tight_layout(); plt.savefig("density.pdf")

In [ ]:
n_end0 = np.argmin(np.abs(nu_arr[0, int(len(x_sweep)/2):-1 ] - .01)) + int(len(x_sweep)/2)
n_end1 = np.argmin(np.abs(nu_arr[-1, int(len(x_sweep)/2):-1 ] - .01)) + int(len(x_sweep)/2)

n_end01 = np.argmin(np.abs(nu_arr[0, int(len(x_sweep)/2):-1 ] - 2.1)) + int(len(x_sweep)/2)
n_end11 = np.argmin(np.abs(nu_arr[-1, int(len(x_sweep)/2):-1 ] - 2.1)) + int(len(x_sweep)/2)

n_end02 = np.argmin(np.abs(nu_arr[0, int(len(x_sweep)/2):-1 ] - 4.1)) + int(len(x_sweep)/2)
n_end12 = np.argmin(np.abs(nu_arr[-1, int(len(x_sweep)/2):-1 ] - 4.1)) + int(len(x_sweep)/2)

print(n_end0)
grad_U = np.gradient(U_arr, x_sweep, axis=-1)
lB = 25.5/B_T**.5
omB = 1.73*B_T

meV_to_THz = 0.24180
factor = lB**2 * meV_to_THz

plt.figure(figsize=(20,4))
plt.plot(x_sweep, factor*abs(grad_U[-1]), lw=3, color='blue')
plt.plot(x_sweep, factor*abs(grad_U[0]), lw=3, color='orange')
plt.vlines(x_sweep[n_end0], 1e-9, 1e3, color='orange', lw=3, ls='--')
plt.vlines(x_sweep[n_end1], 1e-9, 1e3, color='blue', lw=3, ls='--')
plt.vlines(x_sweep[n_end01], 1e-9, 1e3, color='orange', lw=3, ls='--')
plt.vlines(x_sweep[n_end11], 1e-9, 1e3, color='blue', lw=3, ls='--')
plt.vlines(x_sweep[n_end02], 1e-9, 1e3, color='orange', lw=3, ls='--')
plt.vlines(x_sweep[n_end12], 1e-9, 1e3, color='blue', lw=3, ls='--')

plt.ylim(1e-0,1e3)
plt.yscale('log')
plt.grid()
#plt.xlim(3500,5000)

plt.xlabel(r"$x$ - [nm]")
plt.ylabel(r"$|l_B^2 \partial_x U|$ - [Km/s]")


In [ ]:
nu_int = np.trunc(nu_arr[:, int(len(x_sweep)/2)]).astype(int)
nu_int = np.trunc(nu_int/2.).astype(int)*2

sizeW = int(len(nu_arr[0, int(len(x_sweep)/2):-1 ]))
nu_int = np.tile(nu_int[:, np.newaxis], (1,sizeW) )
arr_n_end0 = np.argmin(np.abs(nu_arr[:, int(len(x_sweep)/2):-1 ] - .1), axis=1) + int(len(x_sweep)/2)
arr_n_endL = np.argmin(np.abs(nu_arr[:, 0:int(len(x_sweep)/2) ] - .1), axis=1)

arr_n_end2 = np.argmin(np.abs(nu_arr[:, int(len(x_sweep)/2):-1 ] - nu_int+.01), axis=1) + int(len(x_sweep)/2)

arr_x_edge_R = x_sweep[arr_n_end0]
arr_x_edge_L = x_sweep[arr_n_endL]
size_ne = arr_x_edge_R-arr_x_edge_L
arr_x_IS_R = x_sweep[arr_n_end2]
arr_w = arr_x_edge_R - arr_x_IS_R

plt.plot(dist_values, arr_w, lw=3, color='blue', label=r'$x_{\rm edge}$')
#plt.plot(dist_values, (size_ne-size_ne[-1])/size_ne[-1]*100, lw=3, color='blue', label=r'$x_{\rm edge}$')
plt.xscale('log')


In [ ]:
# Font configuration
font = {
    'family': 'serif',
    'color': 'black',
    'weight': 'normal',
    'size': 16,
}
# Apply the font globally
plt.rcParams['font.family'] = font['family']
plt.rcParams['font.size'] = font['size']
plt.rcParams['axes.labelsize'] = font['size']
plt.rcParams['xtick.labelsize'] = font['size']
plt.rcParams['ytick.labelsize'] = font['size']

central_n = n_arr[:, int(len(x_sweep)/2)]
plt.plot(dist_values, (central_n-central_n[-1])/central_n[-1]*100, lw=3, color='orange', label=r'$\Delta\n_{e}$')
plt.xscale('log')
plt.xlabel(r"$d_g$ - [nm]")
plt.ylabel(r"$\Delta n_0 / n_0$ - [%]")
plt.xlim(1e2,1e4)
plt.grid(True, which="both")

plt.savefig(RESULTS_DIR/"edge_width_sweep.pdf")

In [ ]:
font={'family':'serif','color':'black','weight':'normal','size':16}
plt.rcParams['font.family']=font['family']; plt.rcParams['font.size']=font['size']
plt.rcParams['axes.labelsize']=font['size']; plt.rcParams['xtick.labelsize']=font['size']; plt.rcParams['ytick.labelsize']=font['size']


n_d=np.argmin(np.abs(dist_values-150)); n_fixed=n_arr[n_d]*1e3
plt.figure(figsize=(8,4)); plt.plot(x_sweep,n_fixed,lw=3,color='blue')
plt.xlabel(r"$x$ [nm]"); plt.ylabel(r"$n_e(x)\times 10^{11}$ - [cm$^{-2}$]"); plt.grid(True)
plt.xlim(-p.d,p.d); plt.ylim(0,5); plt.yticks(np.arange(0,5,1)); plt.tight_layout()

## Save numerical data and metadata

In [ ]:
outdir = RESULTS_DIR
outdir.mkdir(parents=True, exist_ok=True)

data_path = outdir / "gate_sweep_data.npz"
meta_path = outdir / "gate_sweep_metadata.json"

params_dict = dataclasses.asdict(p_sweep)
params_dict = {
    k: (
        float(v) if isinstance(v, (np.floating, float))
        else int(v) if isinstance(v, (np.integer, int))
        else v
    )
    for k, v in params_dict.items()
}

np.savez_compressed(
    data_path,
    x=x_sweep,
    dist_values=dist_values,
    U_arr=U_arr,
    n_arr=n_arr,
    nu_arr=nu_arr,
    comp_arr=comp_arr,
    mu_arr=mu_arr,
    U_ext_arr=np.array([r["U_ext"] for r in sweep]),
    x_donor=np.array([r["x_donor"] for r in sweep], dtype=object),
    n_donor_profile=np.array([r["n_donor_profile"] for r in sweep], dtype=object),
    history=np.array([r["history"] for r in sweep], dtype=object),
)

metadata = {
    "description": "Top-gate distance sweep for self-consistent TFA confinement with selectable Coulomb kernel.",
    "kernel_kind": KERNEL_KIND,
    "top_open_box_halfwidth_nm": TOP_OPEN_BOX_HALFWIDTH_NM,
    "neumann_base_halfwidth_nm": NEUMANN_BASE_HALFWIDTH_NM,
    "neumann_depletion_width_nm": NEUMANN_DEPLETION_WIDTH_NM,
    "resolved_neumann_xN_nm": NEUMANN_BASE_HALFWIDTH_NM - NEUMANN_DEPLETION_WIDTH_NM,
    "neumann_surface_charges_enabled": False,
    "neumann_electron_kernel": "top_gate_neumann",
    "neumann_donor_kernel": "top_gate",
    "dist_ref_nm": DIST_REF_NM,
    "sweep": {
        "dist_min_nm": SWEEP_DIST_MIN_NM,
        "dist_max_nm": SWEEP_DIST_MAX_NM,
        "n_points": SWEEP_N_POINTS,
        "max_iter": SWEEP_MAX_ITER,
        "tol": SWEEP_TOL,
        "warm_start": USE_WARM_START,
    },
    "units": {
        "x": "nm",
        "dist_values": "nm",
        "U_arr": "meV",
        "U_ext_arr": "meV",
        "n_arr": "nm^-2",
        "nu_arr": "dimensionless",
        "comp_arr": "nm^-2 meV^-1",
        "mu_arr": "meV",
        "n_donor_profile": "nm^-2",
    },
    "params": params_dict,
}

with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("saved numerical data:", data_path)
print("saved metadata:", meta_path)
print("ffmpeg found at:", shutil.which("ffmpeg"))
print("Current working directory:", Path.cwd())

## Reloading

```python
data = np.load("results/gate_sweep_data.npz", allow_pickle=True)
x_sweep = data["x"]
dist_values = data["dist_values"]
U_arr = data["U_arr"]
n_arr = data["n_arr"]
nu_arr = data["nu_arr"]
comp_arr = data["comp_arr"]
mu_arr = data["mu_arr"]
```